# ArcGIS Data Cleaning

Basic scaffold for cleaning ArcGIS CSV files before any analysis.

Goals:
- load the raw files
- inspect columns and data types
- normalize suburb names and text fields
- handle missing values
- leave the tables ready for export or later joining

In [1]:
# Basic ArcGIS cleaning notebook scaffold
# Load each dataset individually so the notebook stays easy to read and maintain.

from pathlib import Path

import pandas as pd

RAW_ARCGIS_DIR = Path("../data/raw/arcgis")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

suburbs_path = RAW_ARCGIS_DIR / "suburbs.csv"
car_share_path = RAW_ARCGIS_DIR / "Car_share_bays_usage.csv"
libraries_path = RAW_ARCGIS_DIR / "Library_locations.csv"
mobility_parking_path = RAW_ARCGIS_DIR / "Mobility_parking.csv"
sports_path = RAW_ARCGIS_DIR / "Sports_and_recreation_facilities.csv"
playgrounds_path = RAW_ARCGIS_DIR / "Playgrounds.csv"

def load_arcgis_file(path):
    frame = pd.read_csv(path)
    frame.columns = [column.strip() for column in frame.columns]
    return frame

suburbs_df = load_arcgis_file(suburbs_path)
car_share_df = load_arcgis_file(car_share_path)
libraries_df = load_arcgis_file(libraries_path)
mobility_parking_df = load_arcgis_file(mobility_parking_path)
sports_df = load_arcgis_file(sports_path)
playgrounds_df = load_arcgis_file(playgrounds_path)

{
    "suburbs": suburbs_df.shape,
    "car_share": car_share_df.shape,
    "libraries": libraries_df.shape,
    "mobility_parking": mobility_parking_df.shape,
    "sports": sports_df.shape,
    "playgrounds": playgrounds_df.shape,
}

{'suburbs': (657, 10),
 'car_share': (59264, 13),
 'libraries': (11, 9),
 'mobility_parking': (360, 15),
 'sports': (70, 32),
 'playgrounds': (162, 5)}

## 1. Initial loading and inspection

Check file names, columns, data types, and missing values before changing anything.

In [2]:
import re


def normalize_suburb(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if not text:
        return pd.NA
    return " ".join(text.split()).title()


def clean_text_field(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    if not text:
        return pd.NA

    # Remove trailing NSW marker variants such as:
    # "(NSW)", "(nsw)", "(N.S.W.)", "(n.s.w.)"
    text = re.sub(r"\s*\((?:n\.?s\.?w\.?)\)\s*$", "", text, flags=re.IGNORECASE)

    # Remove trailing parenthetical qualifiers ending in "- NSW", e.g.:
    # "Silverwater (Parramatta - NSW)" -> "Silverwater"
    # "(The Hills Shire - NSW)" -> ""
    text = re.sub(r"\s*\([^)]*?-\s*n\.?s\.?w\.?\)\s*$", "", text, flags=re.IGNORECASE)

    text = " ".join(text.split())
    return text if text else pd.NA


def load_arcgis_file(path):
    frame = pd.read_csv(path)
    frame.columns = [column.strip() for column in frame.columns]
    return frame


def preview_frame(frame, name):
    print(f"File: {name}")
    print(f"Rows: {len(frame)}")
    print(f"Columns: {', '.join(frame.columns)}")
    if "Suburb" in frame.columns:
        print("Suburb values:")
        print(frame["Suburb"].dropna().unique())
    print("-" * 40)


preview_frame(suburbs_df, "suburbs.csv")
preview_frame(car_share_df, "Car_share_bays_usage.csv")
preview_frame(libraries_df, "Library_locations.csv")
preview_frame(mobility_parking_df, "Mobility_parking.csv")
preview_frame(sports_df, "Sports_and_recreation_facilities.csv")
preview_frame(playgrounds_df, "Playgrounds.csv")

File: suburbs.csv
Rows: 657
Columns: SAL_CODE21, SAL_NAME21, STE_CODE21, STE_NAME21, AUS_CODE21, AUS_NAME21, AREASQKM21, LOCI_URI21, SHAPE_Leng, SHAPE_Area
----------------------------------------
File: Car_share_bays_usage.csv
Rows: 59264
Columns: OBJECTID, Bay_ID, Street, Suburb, Operator, Status, Date, Year, Month, Hours_Available, Hours_Booked, No_Bookings, Trip_Distance
Suburb values:
['Surry Hills' 'Elizabeth Bay' 'Pyrmont' 'Potts Point' 'Redfern' 'Glebe'
 'Darlinghurst' 'Rushcutters Bay' 'Ultimo' 'Chippendale' 'Woolloomooloo'
 'Haymarket' 'Paddington' 'Millers Point' 'Camperdown' 'Darlington'
 'Newtown' 'Sydney' 'Waterloo' 'Zetland' 'Erskineville' 'Forest Lodge'
 'Alexandria' 'Alexandria ' 'Paddington ' 'Dawes Point' 'Eveleigh'
 'The Rocks' 'Rosebery' 'Haymarket ' 'Centennial Park' 'Moore Park'
 'Beaconsfield' 'Barangaroo' 'Zetland ' 'Bondi' 'ULTIMO' 'ZETLAND'
 'Abbotsford' 'Surry Hills ' 'Rosebery ' 'Forest Loadge']
----------------------------------------
File: Library_locatio

## 2. Base cleaning

### 2.1 Standardize column names

#### 2.1.1 Lowercase, replace spaces with underscores, remove special characters

In [3]:
# Standardize column names
# Lowercase, replace spaces with underscores, remove special characters.

def standardize_column_names(columns):
    standardized = []
    for col in columns:
        col = col.strip().lower().replace(" ", "_")
        col = "".join(c for c in col if c.isalnum() or c == "_")
        standardized.append(col)
    return standardized


def clean_arcgis_data(frame):
    frame = frame.copy()
    frame.columns = standardize_column_names(frame.columns)
    if "suburb" in frame.columns:
        frame["suburb"] = frame["suburb"].apply(normalize_suburb)
    return frame


suburbs_clean = clean_arcgis_data(suburbs_df)
car_share_clean = clean_arcgis_data(car_share_df)
libraries_clean = clean_arcgis_data(libraries_df)
mobility_parking_clean = clean_arcgis_data(mobility_parking_df)
sports_clean = clean_arcgis_data(sports_df)
playgrounds_clean = clean_arcgis_data(playgrounds_df)


print("Cleaned column names:")
{
    "suburbs": suburbs_clean.columns.tolist(),
    "car_share": car_share_clean.columns.tolist(),
    "libraries": libraries_clean.columns.tolist(),
    "mobility_parking": mobility_parking_clean.columns.tolist(),
    "sports": sports_clean.columns.tolist(),
    "playgrounds": playgrounds_clean.columns.tolist(),
}   

Cleaned column names:


{'suburbs': ['sal_code21',
  'sal_name21',
  'ste_code21',
  'ste_name21',
  'aus_code21',
  'aus_name21',
  'areasqkm21',
  'loci_uri21',
  'shape_leng',
  'shape_area'],
 'car_share': ['objectid',
  'bay_id',
  'street',
  'suburb',
  'operator',
  'status',
  'date',
  'year',
  'month',
  'hours_available',
  'hours_booked',
  'no_bookings',
  'trip_distance'],
 'libraries': ['x',
  'y',
  'objectid',
  'name',
  'street_number',
  'street',
  'street_extra',
  'suburb',
  'postcode'],
 'mobility_parking': ['x',
  'y',
  'objectid',
  'address',
  'street',
  'location',
  'suburb',
  'sideofstreet',
  'numberparkingspaces',
  'parkingspacewidth',
  'parkingspacelength',
  'parkingspaceangle',
  'signtext',
  'url',
  'auditdate'],
 'sports': ['x',
  'y',
  'objectid',
  'id',
  'facilityname',
  'address',
  'suburb',
  'postcode',
  'facilitytype',
  'url',
  'status',
  'owner',
  'afl',
  'athletics',
  'badminton',
  'basketball',
  'cityprograms',
  'cricket',
  'futsal',
  '

#### 2.1.2 standardize suburbs csv file column names

In [4]:
suburbs_clean.head()

,sal_code21,sal_name21,ste_code21,ste_name21,aus_code21,aus_name21,areasqkm21,loci_uri21,shape_leng,shape_area
0,10002,Abbotsbury,1,New South Wales,AUS,Australia,4.9788,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.123051,0.000485
1,10003,Abbotsford (NSW),1,New South Wales,AUS,Australia,1.0180,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.053423,0.000099
2,10014,Acacia Gardens,1,New South Wales,AUS,Australia,1.0013,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.044558,0.000097
3,10021,Agnes Banks,1,New South Wales,AUS,Australia,15.4750,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.215383,0.001504
4,10022,Airds,1,New South Wales,AUS,Australia,2.3806,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.071892,0.000233


In [5]:
# replace the 21 in the field name 
suburbs_clean.columns = [col.replace("21", "") for col in suburbs_clean.columns]
suburbs_clean.head()

,sal_code,sal_name,ste_code,ste_name,aus_code,aus_name,areasqkm,loci_uri,shape_leng,shape_area
0,10002,Abbotsbury,1,New South Wales,AUS,Australia,4.9788,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.123051,0.000485
1,10003,Abbotsford (NSW),1,New South Wales,AUS,Australia,1.0180,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.053423,0.000099
2,10014,Acacia Gardens,1,New South Wales,AUS,Australia,1.0013,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.044558,0.000097
3,10021,Agnes Banks,1,New South Wales,AUS,Australia,15.4750,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.215383,0.001504
4,10022,Airds,1,New South Wales,AUS,Australia,2.3806,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.071892,0.000233


In [6]:
# change the name of the field to match the other datasets
suburbs_clean.rename(columns={"sal_name": "suburb"}, inplace=True)
suburbs_clean.head()

,sal_code,suburb,ste_code,ste_name,aus_code,aus_name,areasqkm,loci_uri,shape_leng,shape_area
0,10002,Abbotsbury,1,New South Wales,AUS,Australia,4.9788,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.123051,0.000485
1,10003,Abbotsford (NSW),1,New South Wales,AUS,Australia,1.0180,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.053423,0.000099
2,10014,Acacia Gardens,1,New South Wales,AUS,Australia,1.0013,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.044558,0.000097
3,10021,Agnes Banks,1,New South Wales,AUS,Australia,15.4750,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.215383,0.001504
4,10022,Airds,1,New South Wales,AUS,Australia,2.3806,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.071892,0.000233


### 2.2 Clean text fields


- Explore the fields of the datasets

In [7]:
# Explore the fields of the datasets
# explore the text fields suburb if field exists in dataset.

print("Suburb field values:")
print(suburbs_clean["suburb"].dropna().unique())


Suburb field values:
['Abbotsbury' 'Abbotsford (NSW)' 'Acacia Gardens' 'Agnes Banks' 'Airds'
 'Alexandria' 'Alfords Point' 'Allambie Heights' 'Allawah' 'Ambarvale'
 'Angus' 'Annandale (NSW)' 'Annangrove' 'Arcadia (NSW)' 'Arncliffe'
 'Arndell Park' 'Artarmon' 'Ashbury' 'Ashcroft' 'Ashfield (NSW)' 'Asquith'
 'Auburn (NSW)' 'Austral' 'Avalon Beach' 'Badgerys Creek' 'Balgowlah'
 'Balgowlah Heights' 'Balmain' 'Balmain East' 'Bangor (NSW)' 'Banksia'
 'Banksmeadow' 'Bankstown' 'Bankstown Aerodrome' 'Barangaroo'
 'Barden Ridge' 'Bardia' 'Bardwell Park' 'Bardwell Valley' 'Bass Hill'
 'Baulkham Hills' 'Bayview (NSW)' 'Beacon Hill' 'Beaconsfield (NSW)'
 'Beaumont Hills' 'Beecroft' 'Belfield' 'Bella Vista' 'Bellevue Hill'
 'Belmore' 'Belrose' 'Berala' 'Berkshire Park' 'Berowra' 'Berowra Heights'
 'Berowra Waters' 'Berrilee' 'Beverley Park' 'Beverly Hills' 'Bexley'
 'Bexley North' 'Bidwill (NSW)' 'Bilgola Beach' 'Bilgola Plateau'
 'Birchgrove' 'Birrong' 'Blackett' 'Blacktown' 'Blair Athol (NSW)'
 '

In [8]:
# remove the (NSW) in the suburb names
suburbs_clean["suburb"] = suburbs_clean["suburb"].apply(clean_text_field)


print("Suburb field values:")
print(suburbs_clean["suburb"].dropna().unique())

Suburb field values:
['Abbotsbury' 'Abbotsford' 'Acacia Gardens' 'Agnes Banks' 'Airds'
 'Alexandria' 'Alfords Point' 'Allambie Heights' 'Allawah' 'Ambarvale'
 'Angus' 'Annandale' 'Annangrove' 'Arcadia' 'Arncliffe' 'Arndell Park'
 'Artarmon' 'Ashbury' 'Ashcroft' 'Ashfield' 'Asquith' 'Auburn' 'Austral'
 'Avalon Beach' 'Badgerys Creek' 'Balgowlah' 'Balgowlah Heights' 'Balmain'
 'Balmain East' 'Bangor' 'Banksia' 'Banksmeadow' 'Bankstown'
 'Bankstown Aerodrome' 'Barangaroo' 'Barden Ridge' 'Bardia'
 'Bardwell Park' 'Bardwell Valley' 'Bass Hill' 'Baulkham Hills' 'Bayview'
 'Beacon Hill' 'Beaconsfield' 'Beaumont Hills' 'Beecroft' 'Belfield'
 'Bella Vista' 'Bellevue Hill' 'Belmore' 'Belrose' 'Berala'
 'Berkshire Park' 'Berowra' 'Berowra Heights' 'Berowra Waters' 'Berrilee'
 'Beverley Park' 'Beverly Hills' 'Bexley' 'Bexley North' 'Bidwill'
 'Bilgola Beach' 'Bilgola Plateau' 'Birchgrove' 'Birrong' 'Blackett'
 'Blacktown' 'Blair Athol' 'Blairmount' 'Blakehurst' 'Bondi' 'Bondi Beach'
 'Bondi Juncti

In [9]:
# show the suburb value of the other datasets to check if they match the suburb field in the suburbs dataset. If not, we will need to clean them as well.

car_share_clean["suburb"].dropna().unique()



array(['Surry Hills', 'Elizabeth Bay', 'Pyrmont', 'Potts Point',
       'Redfern', 'Glebe', 'Darlinghurst', 'Rushcutters Bay', 'Ultimo',
       'Chippendale', 'Woolloomooloo', 'Haymarket', 'Paddington',
       'Millers Point', 'Camperdown', 'Darlington', 'Newtown', 'Sydney',
       'Waterloo', 'Zetland', 'Erskineville', 'Forest Lodge',
       'Alexandria', 'Dawes Point', 'Eveleigh', 'The Rocks', 'Rosebery',
       'Centennial Park', 'Moore Park', 'Beaconsfield', 'Barangaroo',
       'Bondi', 'Abbotsford', 'Forest Loadge'], dtype=object)

In [10]:
libraries_clean["suburb"].dropna().unique()


array(['Waterloo', 'Glebe', 'Potts Point', 'Newtown', 'Surry Hills',
       'Sydney', 'Zetland', 'Pyrmont', 'Ultimo', 'Haymarket'],
      dtype=object)

In [11]:
mobility_parking_clean["suburb"].dropna().unique()


array(['Dawes Point', 'Erskineville', 'Haymarket', 'Redfern',
       'Centennial Park', 'Zetland', 'Alexandria', 'Waterloo',
       'Chippendale', 'Beaconsfield', 'St Peters', 'Rushcutters Bay',
       'Potts Point', 'Eveleigh', 'Annandale', 'Camperdown',
       'Darlinghurst', 'Elizabeth Bay', 'Darlington', 'The Rocks',
       'Millers Point', 'Paddington', 'Forest Lodge', 'Newtown',
       'Woolloomooloo', 'Surry Hills', 'Ultimo', 'Pyrmont', 'Glebe',
       'Sydney', 'Surrh Hills', 'Rosebery'], dtype=object)

In [12]:
sports_clean["suburb"].dropna().unique()

array(['Camperdown', 'Surry Hills', 'Newtown', 'Beaconsfield', 'Zetland',
       'Erskineville', 'Moore Park', 'Redfern', 'Broadway',
       'Rushcutters Bay', 'Alexandria', 'Woolloomooloo', 'Waterloo',
       'Glebe', 'Pyrmont', 'Millers Point', 'Sydney', 'Rosebery',
       'The Rocks', 'Ultimo', 'Annandale', 'Barangaroo',
       'Centennial Park', 'Potts Point'], dtype=object)

## 3. Joining and Counting facilities per suburb

In [13]:
# Count facilities per suburb and merge them into the main suburbs table.
# suburbs_clean is the base dataset; the others are secondary sources.

suburbs_base = suburbs_clean.copy()
suburbs_base["suburb"] = suburbs_base["suburb"].apply(normalize_suburb)

secondary_datasets = {
    "car_share_bays": {"frame": car_share_clean, "unique_column": "bay_id"},
    "libraries": {"frame": libraries_clean, "unique_column": None},
    "mobility_parking": {"frame": mobility_parking_clean, "unique_column": None},
    "sports_facilities": {"frame": sports_clean, "unique_column": None},
    "playgrounds": {"frame": playgrounds_clean, "unique_column": None},
}


def count_items_by_suburb(frame, suburb_column="suburb", unique_column=None):
    if suburb_column not in frame.columns:
        return pd.DataFrame(columns=["suburb", "count"])

    working = frame.dropna(subset=[suburb_column]).copy()
    working[suburb_column] = working[suburb_column].apply(normalize_suburb)
    working = working.dropna(subset=[suburb_column])

    if unique_column and unique_column in working.columns:
        counts = (
            working.dropna(subset=[unique_column])
            .groupby(suburb_column)[unique_column]
            .nunique()
            .reset_index(name="count")
        )
    else:
        counts = (
            working.groupby(suburb_column)
            .size()
            .reset_index(name="count")
        )

    return counts


joined_suburbs = suburbs_base.copy()

for dataset_name, config in secondary_datasets.items():
    counts = count_items_by_suburb(config["frame"], unique_column=config["unique_column"])
    counts = counts.rename(columns={"count": f"{dataset_name}_count"})
    joined_suburbs = joined_suburbs.merge(counts, on="suburb", how="left")

count_columns = [column for column in joined_suburbs.columns if column.endswith("_count")]
joined_suburbs[count_columns] = (
    joined_suburbs[count_columns]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)
joined_suburbs["total_facilities"] = joined_suburbs[count_columns].sum(axis=1)

joined_suburbs.head()

,sal_code,suburb,ste_code,ste_name,aus_code,aus_name,areasqkm,loci_uri,shape_leng,shape_area,car_share_bays_count,libraries_count,mobility_parking_count,sports_facilities_count,playgrounds_count,total_facilities
0,10002,Abbotsbury,1,New South Wales,AUS,Australia,4.9788,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.123051,0.000485,0,0,0,0,0,0
1,10003,Abbotsford,1,New South Wales,AUS,Australia,1.0180,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.053423,0.000099,1,0,0,0,0,1
2,10014,Acacia Gardens,1,New South Wales,AUS,Australia,1.0013,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.044558,0.000097,0,0,0,0,0,0
3,10021,Agnes Banks,1,New South Wales,AUS,Australia,15.4750,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.215383,0.001504,0,0,0,0,0,0
4,10022,Airds,1,New South Wales,AUS,Australia,2.3806,http://linked.data.gov.au/dataset/asgsed3/SAL/...,0.071892,0.000233,0,0,0,0,0,0


### 3.1 Visualize total facilities per suburb

A horizontal bar chart works well here because suburb names are easier to read and the ranking is clearer.

In [14]:
facilities_head = (
    joined_suburbs[["sal_code", "suburb", "car_share_bays_count", "libraries_count", "mobility_parking_count", "sports_facilities_count", "total_facilities"]]
    .sort_values("total_facilities", ascending=False)
)

facilities_head

,sal_code,suburb,car_share_bays_count,libraries_count,mobility_parking_count,sports_facilities_count,total_facilities
577,13714,Surry Hills,140,1,29,5,175
249,11645,Glebe,46,1,69,7,123
508,13352,Redfern,71,0,29,5,105
177,11213,Darlinghurst,88,0,12,0,100
500,13297,Pyrmont,54,1,8,3,66
...,...,...,...,...,...,...,...
226,11438,Ermington,0,0,0,0,0
227,11441,Erskine Park,0,0,0,0,0
229,11444,Eschol Park,0,0,0,0,0
231,11480,Fairfield,0,0,0,0,0


In [15]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Simple approach: normalize total_facilities across all suburbs
scaler = MinMaxScaler(feature_range=(0, 100))
facilities_head["facilities_score"] = scaler.fit_transform(facilities_head[["total_facilities"]])

# Weighted approach: if you want to give more importance to certain types
weights = {
    "sports_facilities_count": 0.35,
    "libraries_count":         0.20,
    "car_share_bays_count":    0.10,
    "mobility_parking_count":  0.10,
}

facilities_head["facilities_score"] = sum(
    MinMaxScaler(feature_range=(0,100))
    .fit_transform(facilities_head[[col]]) * w
    for col, w in weights.items()
)

In [16]:
facilities_head

,sal_code,suburb,car_share_bays_count,libraries_count,mobility_parking_count,sports_facilities_count,total_facilities,facilities_score
577,13714,Surry Hills,140,1,29,5,175,49.202899
249,11645,Glebe,46,1,69,7,123,58.285714
508,13352,Redfern,71,0,29,5,105,34.274327
177,11213,Darlinghurst,88,0,12,0,100,8.024845
500,13297,Pyrmont,54,1,8,3,66,30.016563
...,...,...,...,...,...,...,...,...
226,11438,Ermington,0,0,0,0,0,0.000000
227,11441,Erskine Park,0,0,0,0,0,0.000000
229,11444,Eschol Park,0,0,0,0,0,0.000000
231,11480,Fairfield,0,0,0,0,0,0.000000


In [17]:
# Save the cleaned and merged dataset for later use
facilities_head.to_csv(PROCESSED_DIR / "arcgis_suburbs.csv", index=False)